# 1) Import Data
* Import Dependencies
* Import CSV

### Import Dependencies

In [44]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
from mapping import MILITARY_BASES

### Import CSV

In [45]:
data_raw = pd.read_csv("aircraft_data.csv")

# Convert 'timestamp' column to datetime
data_raw["datetime"] = pd.to_datetime(data_raw["datetime"])

# Sort by hex and then datetime
data_raw = data_raw.sort_values(by=["hex", "datetime"]).reset_index(drop=True)

print("Unique hex codes: ", data_raw['hex'].nunique())
print("Unique callsigns: ", data_raw['callsign'].nunique())
data_raw.head(10)

Unique hex codes:  12
Unique callsigns:  13


,hex,callsign,datetime,squawk,altitude,latitude,longitude,type,heading,ground speed,indicated airspeed
0,ae07e5,RCH849,2025-05-24 20:45:53.788690,727.0,32000,36.045235,-78.921791,C-17A Globemaster,38.0,488.0,NaN
1,ae10b5,FAMUS62,2025-05-24 14:56:54.216231,1767.0,2350,38.639809,-76.870155,C-17A Globemaster,2.0,245.0,NaN
2,ae10b5,FAMUS62,2025-05-24 14:57:54.549414,1767.0,2375,38.689087,-76.871643,C-17A Globemaster,353.0,191.0,NaN
3,ae10b5,FAMUS62,2025-05-24 14:58:54.928074,1767.0,1850,38.733562,-76.870102,C-17A Globemaster,2.0,134.0,NaN
4,ae10b5,FAMUS62,2025-05-24 14:59:55.272929,1767.0,1150,38.767384,-76.870499,C-17A Globemaster,2.0,128.0,NaN
5,ae10b5,FAMUS62,2025-05-24 15:00:55.653502,1767.0,225,38.794697,-76.868591,C-17A Globemaster,352.0,116.0,NaN
6,ae10b5,RCH811,2025-05-24 17:16:25.525356,5631.0,14000,39.028473,-77.374016,C-17A Globemaster,322.0,327.0,NaN
7,ae10b5,RCH811,2025-05-24 17:17:25.841614,5631.0,15850,39.120476,-77.401817,C-17A Globemaster,0.0,357.0,NaN
8,ae10b5,RCH811,2025-05-24 17:18:26.196765,5631.0,17400,39.223103,-77.410927,C-17A Globemaster,359.0,389.0,NaN
9,ae10b5,RCH811,2025-05-24 17:19:26.518034,5631.0,19025,39.322002,-77.421875,C-17A Globemaster,354.0,372.0,NaN


# 2) Transform Data
* Base Metrics

### Base Metrics

In [ ]:
max_distance_miles = 15

def find_nearest_base(row):
  """
  Checks if a given lat/lon is within max_distance_miles of any base.
  If so, adds base name and distance from base.
  """
  row['near base'] = None
  row['distance to base'] = None
  point = (row['latitude'], row['longitude'])
  if row['altitude'] <= 10000:
    for base_name, coords in MILITARY_BASES.items():
      base_point = (coords['lat'], coords['lon'])
      distance = geodesic(point, base_point).miles
      if distance <= max_distance_miles:
        row['near base'] = base_name
        row['distance to base'] = int(distance)
        break
  return row

# --- Apply the function to your DataFrame ---
df = data_raw.apply(lambda row: find_nearest_base(row), axis=1)

df.head()


DataFrame with 'near_base' field:


,hex,callsign,datetime,squawk,altitude,latitude,longitude,type,heading,ground speed,indicated airspeed,near base,distance to base
0,ae07e5,RCH849,2025-05-24 20:45:53.788690,727.0,32000,36.045235,-78.921791,C-17A Globemaster,38.0,488.0,NaN,None,NaN
1,ae10b5,FAMUS62,2025-05-24 14:56:54.216231,1767.0,2350,38.639809,-76.870155,C-17A Globemaster,2.0,245.0,NaN,JB Andrews,11.0
2,ae10b5,FAMUS62,2025-05-24 14:57:54.549414,1767.0,2375,38.689087,-76.871643,C-17A Globemaster,353.0,191.0,NaN,JB Andrews,7.0
3,ae10b5,FAMUS62,2025-05-24 14:58:54.928074,1767.0,1850,38.733562,-76.870102,C-17A Globemaster,2.0,134.0,NaN,JB Andrews,4.0
4,ae10b5,FAMUS62,2025-05-24 14:59:55.272929,1767.0,1150,38.767384,-76.870499,C-17A Globemaster,2.0,128.0,NaN,JB Andrews,2.0


In [47]:
test = df[~df['near base'].isna()]

test.head(50)

,hex,callsign,datetime,squawk,altitude,latitude,longitude,type,heading,ground speed,indicated airspeed,near base,distance to base
1,ae10b5,FAMUS62,2025-05-24 14:56:54.216231,1767.0,2350,38.639809,-76.870155,C-17A Globemaster,2.0,245.0,NaN,JB Andrews,11.0
2,ae10b5,FAMUS62,2025-05-24 14:57:54.549414,1767.0,2375,38.689087,-76.871643,C-17A Globemaster,353.0,191.0,NaN,JB Andrews,7.0
3,ae10b5,FAMUS62,2025-05-24 14:58:54.928074,1767.0,1850,38.733562,-76.870102,C-17A Globemaster,2.0,134.0,NaN,JB Andrews,4.0
4,ae10b5,FAMUS62,2025-05-24 14:59:55.272929,1767.0,1150,38.767384,-76.870499,C-17A Globemaster,2.0,128.0,NaN,JB Andrews,2.0
5,ae10b5,FAMUS62,2025-05-24 15:00:55.653502,1767.0,225,38.794697,-76.868591,C-17A Globemaster,352.0,116.0,NaN,JB Andrews,0.0
268,ae145b,RCH813,2025-05-24 15:28:17.916994,3232.0,2350,38.234005,-121.964149,C-17A Globemaster,226.0,172.0,NaN,Travis AFB,2.0
269,ae145b,RCH813,2025-05-24 15:29:18.279966,3232.0,4000,38.195164,-121.965569,C-17A Globemaster,116.0,257.0,NaN,Travis AFB,5.0
270,ae145b,RCH813,2025-05-24 15:30:18.618281,3232.0,5300,38.174294,-121.896873,C-17A Globemaster,114.0,246.0,NaN,Travis AFB,7.0
271,ae145b,RCH813,2025-05-24 15:31:18.971935,3232.0,6850,38.147701,-121.807953,C-17A Globemaster,110.0,289.0,NaN,Travis AFB,11.0
352,ae1469,RCH676,2025-05-24 15:11:59.522178,NaN,1400,43.108322,-70.846657,C-17A Globemaster,NaN,NaN,NaN,Pease ANGB,2.0
